# Joining rent and vacancy data

Both datasets provided by XYMAX in November 2023. Flag attribute indicates which grids had actual numbers (1) and which were geostatistically calculated (0). Since both geojson files have mutual fields, I will need to add table identifiers to the field names before merging them into one.

In [1]:
import pandas as pd
import geopandas as gpd


In [2]:
# import tokyo_slim_pop.geojson and vacancy.geojson
rent = gpd.read_file('rent.geojson')
vacancy = gpd.read_file('vacancy.geojson')

In [3]:
# rename columns with table prefixes and merge on Code/YEAR
# keep geometry from rent and drop geometry from vacancy to avoid duplicates
rent_prefixed = rent.rename(columns={c: f"rent_{c}" for c in rent.columns if c not in ['Code', 'YEAR', 'geometry']})
vacancy_no_geom = vacancy.drop(columns=['geometry'], errors='ignore')
vacancy_prefixed = vacancy_no_geom.rename(columns={c: f"vacancy_{c}" for c in vacancy_no_geom.columns if c not in ['Code', 'YEAR']})

rent_prefixed['YEAR'] = rent_prefixed['YEAR'].astype(str)
vacancy_prefixed['YEAR'] = vacancy_prefixed['YEAR'].astype(str)

rent_prefixed.head(), vacancy_prefixed.head()

merged = rent_prefixed.merge(vacancy_prefixed, on=['Code', 'YEAR'], how='inner')
merged.head()


,Code,rent_max,rent_max_var,rent_FLG,YEAR,rent_mean,rent_mean_var,rent_median,rent_median_var,geometry,vacancy_空室率,vacancy_空室率分散,vacancy_RENTABLE_AREA_NET_SUM,vacancy_KEIYAKUMENSEKITUBO_SUM,vacancy_FLG
0,533925454,12500.000000,7.970208e+07,0.0,2012,11287.500000,3.104316e+07,11500.0,2.992538e+07,"POLYGON ((139.7 35.5375, 139.7 35.54167, 139.6...",0.067770,0.102441,NaN,NaN,0.0
1,533925461,9000.000000,1.062694e+08,0.0,2012,9000.000000,4.139088e+07,9000.0,3.990050e+07,"POLYGON ((139.70625 35.53333, 139.70625 35.537...",0.117405,0.153311,NaN,NaN,0.0
2,533925462,12500.000000,7.970208e+07,0.0,2012,11287.500000,3.104316e+07,11500.0,2.992538e+07,"POLYGON ((139.7125 35.53333, 139.7125 35.5375,...",0.099054,0.115535,NaN,NaN,0.0
3,533925463,12500.000000,7.970208e+07,0.0,2012,11287.500000,3.104316e+07,11500.0,2.992538e+07,"POLYGON ((139.70625 35.5375, 139.70625 35.5416...",0.055977,0.092511,NaN,NaN,0.0
4,533925464,13666.666667,7.084629e+07,0.0,2012,12858.333333,2.759392e+07,13000.0,2.660034e+07,"POLYGON ((139.7125 35.5375, 139.7125 35.54167,...",0.081717,0.092739,NaN,NaN,0.0


In [4]:
# export merged GeoDataFrame
merged.to_file('rent_vacancy.geojson', driver='GeoJSON')